<a href="https://colab.research.google.com/github/pentonfreak/CART498-GenAI/blob/main/CART489_A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook implements the assignment's P+7 technique and a second P+x variation, ensuring that replacements are proper English words.

For each line, GPT-2 predicts the next token after the line with its final word removed. We then select the token at the specified probability rank, applying filtering to ensure it's a valid English word. The purpose of P+x is to generate less common, more unexpected combinations, so 'logical' results here mean grammatically valid English words, though their semantic fit might be surprising.

In [ ]:
import re
from pathlib import Path

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import nltk
from nltk.corpus import words

# Download the English words corpus if not already present
try:
    words.words('en')
except LookupError:
    nltk.download('words')

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Load the GPT-2 language model and tokenizer.
MODEL_NAME = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Create a set of English words for fast lookup (case-insensitive check)
english_words = set(word.lower() for word in words.words())

print("Model loaded on:", device)


PyTorch version: 2.11.0+cu128
CUDA available: True


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded on: cuda


## 1. Original poem

The poem is stored as individual lines because the transformation is performed once per line.

In [ ]:
poem = """One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is."""

lines = poem.splitlines()

print(f"Number of lines: {len(lines)}")
print("\n".join(f"{i+1:2}. {line}" for i, line in enumerate(lines)))


Number of lines: 15
 1. One must have a mind of winter
 2. To regard the frost and the boughs
 3. Of the pine-trees crusted with snow;
 4. And have been cold a long time
 5. To behold the junipers shagged with ice,
 6. The spruces rough in the distant glitter
 7. Of the January sun; and not to think
 8. Of any misery in the sound of the wind,
 9. In the sound of a few leaves,
10. Which is the sound of the land
11. Full of the same wind
12. That is blowing in the same bare place
13. For the listener, who listens in the snow,
14. And, nothing himself, beholds
15. Nothing that is not there and the nothing that is.


## 2. Helper function: separate the final word

The punctuation at the end of a line is kept with the line rather than treated as part of the replacement word. For example, `snow;` becomes the word `snow` plus punctuation `;`.

In [ ]:
def split_last_word(line):
    """Return (prefix, final_word, punctuation).

    Example:
        'with snow;' -> ('with ', 'snow', ';')
    """
    match = re.match(r"^(.*?)([A-Za-zÀ-ÖØ-öø-ÿ'-]+)([^A-Za-zÀ-ÖØ-öø-ÿ'-]*)$", line)
    if not match:
        # If no alphabetic word is found at the end, treat the whole end as punctuation
        # or handle as per the original problem if it implies a word should always be there
        return line, "", ""
    return match.group(1), match.group(2), match.group(3)

for line in lines[:3]:
    print(split_last_word(line))


('One must have a mind of ', 'winter', '')
('To regard the frost and the ', 'boughs', '')
('Of the pine-trees crusted with ', 'snow', ';')


## 3. Get the Nth-highest-probability GPT-2 token

GPT-2 returns a probability for every token in its vocabulary. We sort those probabilities from highest to lowest and select the token at rank `n`.

The rank is **1-indexed**: rank 1 is the most probable token, rank 7 is the seventh-highest-probability token, etc.

In [ ]:
@torch.no_grad()
def nth_prediction(prefix, n):
    """Return GPT-2's nth-highest-probability next token.

    Parameters
    ----------
    prefix : str
        Text before the original line's final word.
    n : int
        1-indexed probability rank.

    Returns
    -------
    token_text : str
        Decoded GPT-2 token.
    probability : float
        Probability assigned to that token.
    token_id : int
        GPT-2 vocabulary ID.
    """
    if n < 1:
        raise ValueError("n must be at least 1")

    # GPT-2 needs at least one token. A leading space is useful for an
    # empty prefix, although none of the poem's prefixes are empty.
    if prefix == "":
        prefix = " "

    inputs = tokenizer(prefix, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}

    outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]
    probabilities = torch.softmax(logits, dim=-1)[0]

    # Sort all vocabulary tokens by probability, descending.
    sorted_probabilities, sorted_ids = torch.sort(
        probabilities, descending=True
    )

    # Handle cases where n is larger than the vocabulary size
    index = min(n - 1, len(sorted_ids) - 1) # Ensure index is within bounds
    token_id = sorted_ids[index].item()
    probability = sorted_probabilities[index].item()
    token_text = tokenizer.decode([token_id])

    return token_text, probability, token_id


## 4. Transform one line

For each line, the original final word is removed. The remaining text becomes GPT-2's context. We then select the requested probability-ranked token and put it in place of the original final word. Original end punctuation is preserved.

To ensure 'proper English words', the function now searches through a larger range of probability ranks until a valid English word is found. If no such word is found after `max_attempts`, the original word is kept to avoid non-English replacements.

In [ ]:
def transform_line(line, rank):
    prefix, original_word, punctuation = split_last_word(line)

    # If no original_word is found (e.g., line ends in punctuation), return original line
    if not original_word:
        return {
            "original": line,
            "prefix": prefix,
            "original_word": original_word,
            "replacement": original_word,
            "punctuation": punctuation,
            "probability": 1.0, # Placeholder
            "token_id": -1,     # Placeholder
            "raw_token": "",    # Placeholder
            "transformed": line,
        }

    current_rank = rank
    max_attempts = 2000  # Significantly increased attempts to find a valid English word
    replacement_token_info = None

    for _ in range(max_attempts):
        token_text, probability, token_id = nth_prediction(prefix, current_rank)
        clean_token = token_text.strip().lower() # Convert to lowercase for dictionary check

        # Check if the token is alphabetic, not empty, and is a common English word
        # Also, check if it's not just a single letter, which are often not meaningful replacements
        if clean_token.isalpha() and len(clean_token) > 1 and clean_token in english_words:
            replacement_token_info = (token_text, probability, token_id)
            break
        current_rank += 1

    if replacement_token_info is None:
        # Fallback: If after max_attempts no suitable English word is found, keep the original word.
        print(f"Warning: No suitable English word found for '{original_word}' after {max_attempts} attempts. Keeping original word.")
        replacement = original_word
        token_text = original_word # For logging purposes
        probability = 0.0          # Indicating no GPT-2 replacement found
        token_id = -1              # Indicating no GPT-2 replacement found
    else:
        token_text, probability, token_id = replacement_token_info
        # GPT-2's decoded token may contain a leading space. Since the
        # replacement happens after the prefix, remove only that leading
        # whitespace so we do not create a double space.
        replacement = token_text.lstrip()

        # If the selected token decodes to nothing after whitespace removal,
        # use the original raw token to ensure something is there, though this
        # case should be rare with the dictionary filter.
        if replacement == "":
            replacement = token_text

    # Ensure the replacement word retains its original capitalization if possible
    # or at least a reasonable capitalization based on its position.
    # For simplicity, we'll use title case if it was originally capitalized.
    if original_word and original_word[0].isupper():
        replacement = replacement.capitalize()

    transformed = prefix + replacement + punctuation

    return {
        "original": line,
        "prefix": prefix,
        "original_word": original_word,
        "replacement": replacement,
        "punctuation": punctuation,
        "probability": probability,
        "token_id": token_id,
        "raw_token": token_text,
        "transformed": transformed,
    }


## 5. Transform the complete poem

In [ ]:
def transform_poem(lines, rank):
    """Transform every line using one probability rank."""
    return [transform_line(line, rank) for line in lines]


def print_transformation(results):
    for i, result in enumerate(results, start=1):
        print(f"{i:2}. {result['transformed']}")


## 6. Generate P+7

This is the required P+7 version: the **7th-highest-probability token** (filtered for English words) replaces the final word of every line.

In [ ]:
P7_RANK = 7

p7_results = transform_poem(lines, P7_RANK)

print("P+7 transformation:\n")
print_transformation(p7_results)


P+7 transformation:

 1. One must have a mind of ers
 2. To regard the frost and the ers
 3. Of the pine-trees crusted with vern;
 4. And have been cold a long iced
 5. To behold the junipers shagged with vern,
 6. The spruces rough in the distant ers
 7. Of the January sun; and not to ich
 8. Of any misery in the sound of the ich,
 9. In the sound of a few ich,
10. Which is the sound of the ich
11. Full of the same ich
12. That is blowing in the same bare ers
13. For the listener, who listens in the urn,
14. And, nothing himself, iced
15. Nothing that is not there and the nothing that iced.


## 7. Choose the P+x value

For the second version, you can experiment with several ranks. A rank farther down the probability list will generally produce less expected words, which can create more unusual or absurd combinations. Remember, 'logical' here refers to being a valid English word, not necessarily semantically coherent with the original line, as the goal is creative disruption.

The example below uses **12**. After running it, inspect the result and change `X_RANK` if another value produces a funnier/wittier/absurder poem.

In [ ]:
X_RANK = 12

px_results = transform_poem(lines, X_RANK)

print(f"P+{X_RANK} transformation:\n")
print_transformation(px_results)


P+12 transformation:

 1. One must have a mind of ich
 2. To regard the frost and the ers
 3. Of the pine-trees crusted with ers;
 4. And have been cold a long ike
 5. To behold the junipers shagged with ers,
 6. The spruces rough in the distant ides
 7. Of the January sun; and not to urs
 8. Of any misery in the sound of the urs,
 9. In the sound of a few urs,
10. Which is the sound of the ike
11. Full of the same ich
12. That is blowing in the same bare ich
13. For the listener, who listens in the ich,
14. And, nothing himself, ich
15. Nothing that is not there and the nothing that iced.


## 8. Optional: compare several x values

This makes it easier to choose an `x` based on the actual outputs. There is no mathematical requirement that the largest rank is funniest; the choice is an artistic one.

In [ ]:
candidate_ranks = [10, 15, 20, 23, 25, 30, 40]

for rank in candidate_ranks:
    print("=" * 70)
    print(f"P+{rank}")
    print("=" * 70)
    results = transform_poem(lines, rank)
    print_transformation(results)
    print()


P+10
 1. One must have a mind of ers
 2. To regard the frost and the ers
 3. Of the pine-trees crusted with ers;
 4. And have been cold a long ike
 5. To behold the junipers shagged with ers,
 6. The spruces rough in the distant ides
 7. Of the January sun; and not to ers
 8. Of any misery in the sound of the urs,
 9. In the sound of a few ers,
10. Which is the sound of the ute
11. Full of the same ich
12. That is blowing in the same bare ers
13. For the listener, who listens in the ich,
14. And, nothing himself, ich
15. Nothing that is not there and the nothing that iced.

P+15
 1. One must have a mind of urn
 2. To regard the frost and the ice
 3. Of the pine-trees crusted with ers;
 4. And have been cold a long ich
 5. To behold the junipers shagged with ers,
 6. The spruces rough in the distant ides
 7. Of the January sun; and not to urs
 8. Of any misery in the sound of the ers,
 9. In the sound of a few ike,
10. Which is the sound of the ike
11. Full of the same ich
12. That is b

## 9. Save the two required `.txt` files

The filenames are generated automatically. If you change `X_RANK`, the second filename changes with it.

In [ ]:
OUTPUT_DIR = Path(".")

p7_path = OUTPUT_DIR / "P+7.txt"
px_path = OUTPUT_DIR / f"P+{X_RANK}.txt"

p7_text = "\n".join(result["transformed"] for result in p7_results) + "\n"
px_text = "\n".join(result["transformed"] for result in px_results) + "\n"

p7_path.write_text(p7_text, encoding="utf-8")
px_path.write_text(px_text, encoding="utf-8")

print(f"Saved: {p7_path.resolve()}")
print(f"Saved: {px_path.resolve()}")


## 10. Optional verification

The following cell prints the original final word, the selected GPT-2 token (decoded text), its probability, and the resulting line. This is useful for documenting that the transformation really uses the requested probability rank (or the closest valid English word found).

In [ ]:
print("P+7 verification")
print("-" * 90)

for i, result in enumerate(p7_results, start=1):
    print(
        f"{i:2}. original={result['original_word']!r:15} "
        f"replacement={result['replacement']!r:15} "
        f"probability={result['probability']:.8f}"
    )


## Submission checklist

- `P+7.txt` — required seventh-ranked transformation.
- `P+X.txt` (where `X` is your chosen rank) — second transformation using the chosen `x` value.
- This notebook (`.ipynb`) — contains the GPT-2 loading, probability ranking, transformations, and file-writing code.

If another `X` produces a more interesting result, change `X_RANK`, rerun the relevant cells, and submit the resulting `P+X.txt` file.

In [ ]:
import re
from pathlib import Path

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import nltk # Added for dictionary check
from nltk.corpus import words # Added for dictionary check

# Download the English words corpus if not already present
try:
    words.words('en')
except LookupError:
    nltk.download('words')

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
# Load the GPT-2 language model and tokenizer.
# 'gpt2' is the standard GPT-2 model from Hugging Face.

MODEL_NAME = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Create a set of English words for fast lookup
english_words = set(words.words())

print("Model loaded on:", device)
"""## 1. Original poem

The poem is stored as individual lines because the transformation is performed once per line.
"""
poem = """One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is."""

lines = poem.splitlines()

print(f"Number of lines: {len(lines)}")
print("\n".join(f"{i+1:2}. {line}" for i, line in enumerate(lines)))
"""## 2. Helper function: separate the final word

The punctuation at the end of a line is kept with the line rather than treated as part of the replacement word. For example, `snow;` becomes the word `snow` plus punctuation `;`.
"""
def split_last_word(line):
    """Return (prefix, final_word, punctuation).

    Example:
        'with snow;' -> ('with ', 'snow', ';')
    """
    match = re.match(r"^(.*?)([A-Za-zÀ-ÖØ-öø-ÿ'-]+)([^A-Za-zÀ-ÖØ-öø-ÿ'-]*)$", line)
    if not match:
        raise ValueError(f"Could not identify the final word in: {line!r}")
    return match.group(1), match.group(2), match.group(3)


for line in lines[:3]:
    print(split_last_word(line))
"""## 3. Get the Nth-highest-probability GPT-2 token

GPT-2 returns a probability for every token in its vocabulary. We sort those probabilities from highest to lowest and select the token at rank `n`.

The rank is **1-indexed**: rank 1 is the most probable token, rank 7 is the seventh-highest-probability token, etc.
"""
@torch.no_grad()
def nth_prediction(prefix, n):
    """Return GPT-2's nth-highest-probability next token.

    Parameters
    ----------
    prefix : str
        Text before the original line's final word.
    n : int
        1-indexed probability rank.

    Returns
    -------
    token_text : str
        Decoded GPT-2 token.
    probability : float
        Probability assigned to that token.
    token_id : int
        GPT-2 vocabulary ID.
    """
    if n < 1:
        raise ValueError("n must be at least 1")

    # GPT-2 needs at least one token. A leading space is useful for an
    # empty prefix, although none of the poem's prefixes are empty.
    if prefix == "":
        prefix = " "

    inputs = tokenizer(prefix, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}

    outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]
    probabilities = torch.softmax(logits, dim=-1)[0]

    # Sort all vocabulary tokens by probability, descending.
    sorted_probabilities, sorted_ids = torch.sort(
        probabilities, descending=True
    )

    index = n - 1
    token_id = sorted_ids[index].item()
    probability = sorted_probabilities[index].item()
    token_text = tokenizer.decode([token_id])

    return token_text, probability, token_id
"""## 4. Transform one line

For each line, the original final word is removed. The remaining text becomes GPT-2's context. We then select the requested probability-ranked token and put it in place of the original final word. Original end punctuation is preserved.
"""
def transform_line(line, rank):
    prefix, original_word, punctuation = split_last_word(line)

    current_rank = rank
    max_attempts = 200 # Increased attempts as filtering will be stricter
    replacement_token_info = None

    for _ in range(max_attempts):
        token_text, probability, token_id = nth_prediction(prefix, current_rank)
        clean_token = token_text.strip().lower() # Convert to lowercase for dictionary check

        # Check if the token is alphabetic, not empty, and is a common English word
        if clean_token.isalpha() and clean_token and clean_token in english_words:
            replacement_token_info = (token_text, probability, token_id)
            break
        current_rank += 1

    if replacement_token_info is None:
        # Fallback: if after max_attempts no suitable word is found, revert to previous logic
        # using the token at the original requested rank, regardless of dictionary status.
        # This prevents lines from being untransformed if no suitable word is found.
        token_text, probability, token_id = nth_prediction(prefix, rank)
        replacement_token_info = (token_text, probability, token_id)

    token_text, probability, token_id = replacement_token_info

    # GPT-2's decoded token may contain a leading space. Since the
    # replacement happens after the prefix, remove only that leading
    # whitespace so we do not create a double space.
    replacement = token_text.lstrip()

    # If the selected token decodes to nothing after whitespace removal,
    # keep the raw decoded token rather than silently choosing another rank.
    if replacement == "":
        replacement = token_text

    transformed = prefix + replacement + punctuation

    return {
        "original": line,
        "prefix": prefix,
        "original_word": original_word,
        "replacement": replacement,
        "punctuation": punctuation,
        "probability": probability,
        "token_id": token_id,
        "raw_token": token_text,
        "transformed": transformed,
    }
"""## 5. Transform the complete poem
"""
def transform_poem(lines, rank):
    """Transform every line using one probability rank."""
    return [transform_line(line, rank) for line in lines]


def print_transformation(results):
    for i, result in enumerate(results, start=1):
        print(f"{i:2}. {result['transformed']}")
"""## 6. Generate P+7

This is the required P+7 version: the **7th-highest-probability token** replaces the final word of every line.
"""
P7_RANK = 7

p7_results = transform_poem(lines, P7_RANK)

print("P+7 transformation:\n")
print_transformation(p7_results)
"""## 7. Choose the P+x value

For the second version, you can experiment with several ranks. A rank farther down the probability list will generally produce less expected words, which can create more unusual or absurd combinations.

The example below uses **23**. After running it, inspect the result and change `X_RANK` if another value produces a funnier/wittier/absurder poem.
"""
X_RANK = 12

px_results = transform_poem(lines, X_RANK)

print(f"P+{X_RANK} transformation:\n")
print_transformation(px_results)
"""## 8. Optional: compare several x values

This makes it easier to choose an `x` based on the actual outputs. There is no mathematical requirement that the largest rank is funniest; the choice is an artistic one.
"""
candidate_ranks = [10, 15, 20, 23, 25, 30, 40]

for rank in candidate_ranks:
    print("=" * 70)
    print(f"P+{rank}")
    print("=" * 70)
    results = transform_poem(lines, rank)
    print_transformation(results)
    print()
"""## 9. Save the two required `.txt` files

The filenames are generated automatically. If you change `X_RANK`, the second filename changes with it.
"""
OUTPUT_DIR = Path(".")

p7_path = OUTPUT_DIR / "P+7.txt"
px_path = OUTPUT_DIR / f"P+{X_RANK}.txt"

p7_text = "\n".join(result["transformed"] for result in p7_results) + "\n"
px_text = "\n".join(result["transformed"] for result in px_results) + "\n"

p7_path.write_text(p7_text, encoding="utf-8")
px_path.write_text(px_text, encoding="utf-8")

print(f"Saved: {p7_path.resolve()}")
print(f"Saved: {px_path.resolve()}")
"""## 10. Optional verification

The following cell prints the original final word, the selected GPT-2 token, its probability, and the resulting line. This is useful for documenting that the transformation really uses the requested probability rank.
"""
print("P+7 verification")
print("-" * 90)

for i, result in enumerate(p7_results, start=1):
    print(
        f"{i:2}. original={result['original_word']!r:15} "
        f"replacement={result['replacement']!r:15} "
        f"probability={result['probability']:.8f}"
    )
"""## Submission checklist

- `P+7.txt` — required seventh-ranked transformation.
- `P+23.txt` (or another `P+x.txt`) — second transformation using the chosen `x` value.
- This notebook (`.ipynb`) — contains the GPT-2 loading, probability ranking, transformations, and file-writing code.

If another `x` produces a more interesting result, change `X_RANK`, rerun the relevant cells, and submit the resulting `P+x.txt` file.
"""

PyTorch version: 2.11.0+cu128
CUDA available: True


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded on: cuda
Number of lines: 15
 1. One must have a mind of winter
 2. To regard the frost and the boughs
 3. Of the pine-trees crusted with snow;
 4. And have been cold a long time
 5. To behold the junipers shagged with ice,
 6. The spruces rough in the distant glitter
 7. Of the January sun; and not to think
 8. Of any misery in the sound of the wind,
 9. In the sound of a few leaves,
10. Which is the sound of the land
11. Full of the same wind
12. That is blowing in the same bare place
13. For the listener, who listens in the snow,
14. And, nothing himself, beholds
15. Nothing that is not there and the nothing that is.
('One must have a mind of ', 'winter', '')
('To regard the frost and the ', 'boughs', '')
('Of the pine-trees crusted with ', 'snow', ';')
P+7 transformation:

 1. One must have a mind of ers
 2. To regard the frost and the ers
 3. Of the pine-trees crusted with ers;
 4. And have been cold a long iced
 5. To behold the junipers shagged with ers,
 6. The spr

'## Submission checklist\n\n- `P+7.txt` — required seventh-ranked transformation.\n- `P+23.txt` (or another `P+x.txt`) — second transformation using the chosen `x` value.\n- This notebook (`.ipynb`) — contains the GPT-2 loading, probability ranking, transformations, and file-writing code.\n\nIf another `x` produces a more interesting result, change `X_RANK`, rerun the relevant cells, and submit the resulting `P+x.txt` file.\n'

In [19]:
%pip install -q transformers torch

from pathlib import Path
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.eval()

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model:", MODEL_NAME)
print("Device:", device)

text = """One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is."""

lines = text.splitlines()

print("Number of lines:", len(lines))

def replace_final_word(line, rank):
    """
    Replace the final word of a line with the token ranked
    'rank' according to GPT-2's next-token probability.
    """

    # Separate the line into:
    # prefix + whitespace + final word
    match = re.match(r"^(.*?)(\s+)(\S+)(\s*)$", line)

    if not match:
        return line

    prefix, whitespace, original_word, trailing = match.groups()

    # Tokenize everything before the final word
    inputs = tokenizer(
        prefix,
        return_tensors="pt"
    ).to(device)

    # Get GPT-2 predictions
    with torch.no_grad():
        outputs = model(**inputs)

    # GPT-2 predicts the next token
    logits = outputs.logits[:, -1, :]

    # Convert logits to probabilities
    probabilities = torch.softmax(logits, dim=-1)

    # Get the highest-probability tokens
    top_probs, top_ids = torch.topk(
        probabilities,
        k=rank
    )

    # rank=7 means choose the 7th-highest probability
    selected_id = top_ids[0, rank - 1].item()

    # Convert GPT-2 token back into text
    selected_token = tokenizer.decode([selected_id])

    # Remove unwanted line breaks
    selected_token = selected_token.replace("\n", " ")
    selected_token = selected_token.replace("\r", " ")
    selected_token = selected_token.strip()

    # If GPT-2 somehow returns an empty token,
    # keep the original word.
    if not selected_token:
        selected_token = original_word

    return prefix + whitespace + selected_token + trailing


def transform_poem(text, rank):
    """
    Apply the P+rank transformation to every line.
    """

    transformed_lines = []

    for line in text.splitlines():
        transformed_line = replace_final_word(line, rank)
        transformed_lines.append(transformed_line)

    return "\n".join(transformed_lines)


p7 = transform_poem(text, rank=7)

print("P+7")
print("=" * 50)
print(p7)

X = 40

px = transform_poem(text, rank=X)

print(f"P+{X}")
print("=" * 50)
print(px)

# Create the A02 folder
output_dir = Path("A02")
output_dir.mkdir(exist_ok=True)

# Save P+7
p7_file = output_dir / "P+7.txt"
p7_file.write_text(p7, encoding="utf-8")

# Save P+23
px_file = output_dir / f"P+{X}.txt"
px_file.write_text(px, encoding="utf-8")

print("Files saved:")
print(p7_file)
print(px_file)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model: gpt2
Device: cuda
Number of lines: 15
P+7
One must have a mind of her
To regard the frost and the death
Of the pine-trees crusted with oil
And have been cold a long way
To behold the junipers shagged with white
The spruces rough in the distant horizon
Of the January sun; and not to have
Of any misery in the sound of the sound
In the sound of a few shots
Which is the sound of the voice
Full of the same day
That is blowing in the same bare air
For the listener, who listens in the morning
And, nothing himself, I
Nothing that is not there and the nothing that isn
P+40
One must have a mind of consistency
To regard the frost and the effect
Of the pine-trees crusted with seeds
And have been cold a long Time
To behold the junipers shagged with thick
The spruces rough in the distant years
Of the January sun; and not to him
Of any misery in the sound of the old
In the sound of a few notes
Which is the sound of the ball
Full of the same order
That is blowing in the same bare hole
For the l